In [88]:
!pip install -q youtube-transcript-api transformers torch

In [89]:
import re
import torch
from youtube_transcript_api import YouTubeTranscriptApi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [90]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {DEVICE}")

[INFO] Using device: cuda


In [91]:
def extract_video_id(youtube_url: str) -> str:
    regex = r"(?:v=|\/)([0-9A-Za-z_-]{11})"
    match = re.search(regex, youtube_url)
    if match:
      return match.group(1)
    else:
      raise ValueError("Invalid YouTube URL")

In [92]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound, VideoUnavailable

def fetch_youtube_transcript(video_url: str) -> str:
    video_id = extract_video_id(video_url)
    print(f'INFO Extracting transcript for video ID: {video_id}')

    try:
        # New instance-based API usage
        ytt_api = YouTubeTranscriptApi()
        fetched_transcript = ytt_api.fetch(video_id)

        # Join snippets into a single text block
        full_transcript = " ".join(snippet.text for snippet in fetched_transcript)
        print(f"[SUCCESS] Transcript fetched successfully ({len(full_transcript.split())} words).")
        return full_transcript
    except (TranscriptsDisabled, NoTranscriptFound, VideoUnavailable) as e:
        raise RuntimeError(f'YouTube API specific error: {e}')
    except Exception as e:
        raise RuntimeError(f'Unexpected error retrieving transcript: {e}')

In [93]:
MODEL_NAME = "facebook/bart-large-cnn"
print(f"\n[INFO] Loading open-source LLM: {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to(DEVICE)

print(f"Model and Tokenizer loaded successfully on {DEVICE}")


[INFO] Loading open-source LLM: facebook/bart-large-cnn...


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Model and Tokenizer loaded successfully on cuda


In [94]:
def chunk_test(text:str, tokenizer, max_tokens:int=900):
  words = text.split()
  chunks = []
  current_chunk_words = []

  for word in words:
    current_chunk_words.append(word)
    candidate = " ".join(current_chunk_words)
    token_len = len(tokenizer(candidate, add_special_tokens=False).input_ids)

    if token_len >= max_tokens:
      current_chunk_words.pop()
      chunks.append(" ".join(current_chunk_words))
      current_chunk_words = [word]

  if current_chunk_words:
      chunks.append(" ".join(current_chunk_words))

  return chunks

In [95]:
def summarize_chunk(text: str, max_new_tokens=300, min_new_tokens=100) -> str:
  inputs = tokenizer(
      text,
      return_tensors='pt',
      truncation=True,
      max_length=1024
  ).to(DEVICE)
  output_ids = model.generate(
      inputs.input_ids,
      num_beams=4,
      max_new_tokens=max_new_tokens,
      min_new_tokens=min_new_tokens,
      no_repeat_ngram_size=3,
      length_penalty=1.5,   # >1 rewards longer sequences, <1 shortens them
      early_stopping=True
  )
  return tokenizer.decode(output_ids[0], skip_special_tokens=True)

In [96]:
def generate_engineered_summary(transcript_text: str) -> str:
  chunks = chunk_test(transcript_text, tokenizer)
  print(f"[INFO] Transcript split into {len(chunks)} chunk(s) for summarization.")

  chunk_summaries = []
  for i, chunk in enumerate(chunks, start=1):
    print(f"Summarizing Chunk {i}/{len(chunks)}...")
    chunk_summaries.append(summarize_chunk(chunk, max_new_tokens=250, min_new_tokens=80))

  if len(chunk_summaries) == 1:
    return chunk_summaries[0]

  combined = " ".join(chunk_summaries)
  print("[INFO] Producing final consolidated summary...")
  return summarize_chunk(combined, max_new_tokens=500, min_new_tokens=200)

In [97]:
if __name__ == "__main__":
  sample_youtube_url = "https://www.youtube.com/watch?v=AZhKtU_E-Fs"

  try:
      raw_transcript = fetch_youtube_transcript(sample_youtube_url)
      structured_summary = generate_engineered_summary(raw_transcript)
      print("\n" + "=" * 70)
      print("YOUTUBE VIDEO SUMMARIZER — OUTPUT REPORT")
      print("=" * 70)
      print(f"URL: {sample_youtube_url}\n")
      print(f"GENERATED SUMMARY:\n{structured_summary}")
      print("=" * 70)

  except Exception as error:
      print(f"\n[ERROR] Process failed: {error}")

INFO Extracting transcript for video ID: AZhKtU_E-Fs
[SUCCESS] Transcript fetched successfully (1166 words).


[transformers] Both `max_new_tokens` (=250) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=80) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] Transcript split into 2 chunk(s) for summarization.
Summarizing Chunk 1/2...


[transformers] Both `max_new_tokens` (=250) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=80) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Summarizing Chunk 2/2...


[transformers] Both `max_new_tokens` (=500) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=200) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] Producing final consolidated summary...

YOUTUBE VIDEO SUMMARIZER — OUTPUT REPORT
URL: https://www.youtube.com/watch?v=AZhKtU_E-Fs

GENERATED SUMMARY:
The real benefit of chess isn't memorizing openings, it's teaching your brain how to think. Strong players learn to let go of the last move and focus entirely on the next one. When your first plan fails, when you make a mistake, chess quietly teaches you to stop dwelling on what you can't change. What's the biggest lesson chess has taught you that had nothing to do with winning? Let me know in the comments. Back to Mail Online home. Back To the page you came from. Click here to read the rest of the article. Back into the page where you camefrom. Click HERE to see the full transcript of the interview with William Chase, the author of the book, "Chess: The Game That Made Me A Better Chess Player" and the book's co-host, John Defterios. The book is published by Simon & Schuster, and is available in paperback and on Kindle. For more, 